In [ ]:
import os
import re
import json
import pymupdf                       # PyMuPDF

MODEL_FILE = "gemma-3-270m-it-q4_0.gguf"
BEFORE = 80                          # characters of text taken before each date
AFTER = 12                           # characters of text taken after each date (just the time, not the next label)
MAX_PER_CALL = 10                    # candidate lines per model call
STOP_AT_OTHER_DATES = True           # a line never includes another date's text

# --- page filter: does this page have a deadline word at all? -------------------
KEYWORD_RE = re.compile(
    r"\b(due|deadline|no later than|sealed|received|submitted|closing|opening)\b",
    re.I,
)

MON = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*"
DATE_RE = re.compile(
    rf"\b{MON}\.?\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{4}}\b"      # November 5, 2026 / Nov. 5, 2026
    rf"|\b\d{{1,2}}[-\s]{MON}\.?[-\s,]+\d{{4}}\b"                  # 07-Oct-2026 / 5 November 2026
    r"|\b\d{1,2}/\d{1,2}/\d{2,4}\b",                               # 11/05/2026
    re.I,
)

# --- model prompt ---------------------------------------------------------------
INSTRUCTIONS = """Each line below is a piece of text from a procurement document and contains one date.
Find the line that has a keyword showing its date is the deadline for submitting bids, proposals or responses.
Keywords for the deadline look like: submission deadline, closing date, bids due, proposals are due, sealed bids must be submitted no later than, must be received by.
Do not choose a date for publication, issue, addendum, meeting, conference, questions, bid opening or award.
Answer with the keyword you found and that line's date copied exactly as written. If no line shows the submission deadline, answer none.
Answer with JSON only."""

EXAMPLES = [
    (["Publication Date/Time: 9/10/2026 10:00 AM",
      "10:00 AM Closing Date/Time: 10/15/2026 11:00 AM"],
     {"keyword": "Closing Date/Time", "due_date": "10/15/2026"}),
    (["A mandatory pre-bid conference will be held October 9, 2026 at 10:00 a.m.",
      "Sealed proposals must be received no later than 2:00 p.m. on March 4, 2027 at the Purchasing Office.",
      "The contract will be awarded on or about April 12, 2027."],
     {"keyword": "must be received no later than", "due_date": "March 4, 2027"}),
    (["Addendum #1 issued on October 2, 2026.",
      "Questions must be sent in writing by February 12, 2027."],
     {"keyword": "", "due_date": "none"}),
    (["Bids are due Friday, May 7, 2027 at 2:00 PM.",
      "The bid opening will be held May 10, 2027 at 2:10 PM."],
     {"keyword": "Bids are due", "due_date": "May 7, 2027"}),
]

def build_messages(lines: list) -> list:
    msgs = []
    for i, (ex_lines, answer) in enumerate(EXAMPLES):
        content = "\n".join(ex_lines)
        if i == 0:
            content = f"{INSTRUCTIONS}\n\n{content}"        # Gemma has no system role, so the rules go in the first turn
        msgs.append({"role": "user", "content": content})
        msgs.append({"role": "assistant", "content": json.dumps(answer)})
    msgs.append({"role": "user", "content": "\n".join(lines)})
    return msgs

_llm = None

def get_llm():
    """Loads the model once per process (the first call is the slow one)."""
    global _llm
    if _llm is None:
        from llama_cpp import Llama
        path = os.getenv("DUE_DATE_MODEL_PATH")
        if not path:
            try:
                base = os.path.dirname(os.path.abspath(__file__))     # running as a .py file
                path = os.path.join(base, "..", "models", MODEL_FILE)
            except NameError:                                         # running in a notebook
                path = os.path.join(os.getcwd(), "models", MODEL_FILE)
        _llm = Llama(model_path=path, n_ctx=2048, n_threads=os.cpu_count(), verbose=False)
    return _llm

def ask_model(lines: list, dates: list) -> dict:
    """Returns {'keyword': ..., 'due_date': ...}; due_date is one of the dates in `dates`, or 'none'."""
    schema = {
        "type": "object",
        "properties": {
            "keyword": {"type": "string"},
            "due_date": {"type": "string", "enum": dates + ["none"]},   # can only answer with a date from the lines
        },
        "required": ["keyword", "due_date"],
    }
    out = get_llm().create_chat_completion(
        messages=build_messages(lines),
        response_format={"type": "json_object", "schema": schema},
        temperature=0.0,
        max_tokens=60,
    )
    try:
        return json.loads(out["choices"][0]["message"]["content"])
    except (ValueError, KeyError, IndexError):
        return {"keyword": "", "due_date": "none"}

# --- helpers --------------------------------------------------------------------
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.replace("\xa0", " ")).strip()

NEGATIVE_RE = re.compile(
    r"\b(pre-?bid|pre-?proposal|site visit|walk-?through|questions?|inquir\w+|"
    r"rfi|issued|posted|publication|publish\w*|advertis\w+|award\w*)\b",
    re.I,
)

def is_other_date(before: str) -> bool:
    """True only if a negative word is the closest cue before the date (closer than any deadline keyword)."""
    neg = [m.start() for m in NEGATIVE_RE.finditer(before)]
    if not neg:
        return False
    kw = [m.start() for m in KEYWORD_RE.finditer(before)]
    return not kw or neg[-1] > kw[-1]

def date_candidates(text: str) -> list:
    """(date, text before the date, whole line) for every date on the page."""
    matches = list(DATE_RE.finditer(text))
    found = []
    for i, m in enumerate(matches):
        lo = max(0, m.start() - BEFORE)
        hi = min(len(text), m.end() + AFTER)
        if STOP_AT_OTHER_DATES:
            if i > 0:
                lo = max(lo, matches[i - 1].end())
            if i + 1 < len(matches):
                hi = min(hi, matches[i + 1].start())
        before = text[lo:m.start()]
        if lo > 0 and lo == m.start() - BEFORE:
            before = before.split(" ", 1)[-1]
        after = text[m.end():hi]
        if hi == m.end() + AFTER and hi < len(text):
            after = after.rsplit(" ", 1)[0]
        found.append((m.group(0), norm(before), norm(f"{before}{m.group(0)}{after}")))
    return found

# --- main function --------------------------------------------------------------
def find_bid_due_date(pdf_path: str) -> dict:
    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            text = norm(page.get_text())
            if not text or not KEYWORD_RE.search(text):
                continue

            # drop dates where a negative word is the nearest cue before the date
            candidates = [(d, line) for d, before, line in date_candidates(text)
                          if not is_other_date(before)]

            for i in range(0, len(candidates), MAX_PER_CALL):
                chunk = candidates[i:i + MAX_PER_CALL]
                lines = [line for _, line in chunk]
                dates = list(dict.fromkeys(d for d, _ in chunk))
                answer = ask_model(lines, dates)

                due = answer.get("due_date")
                keyword = norm(answer.get("keyword", "")).lower()

                # accept only if the keyword really appears in a line that holds that date
                if due in dates and keyword and any(due in ln and keyword in ln.lower() for ln in lines):
                    return {"bid_due_date": due, "found": "Yes", "keyword": answer["keyword"]}

    return {"bid_due_date": "Not Specified", "found": "No"}

In [2]:
print("is_due_date" in globals(), "ask_model" in globals())    # should print: True False

False True


In [26]:
pdf_path = "/home/upnup/Desktop/kabil/SMI/University_of_Maine_System_Purchasing_Dept/download/2027-002-RFQ-PS-Piano-Tuning-Services.pdf"

In [27]:
res = find_bid_due_date(pdf_path)

In [28]:
res


{'bid_due_date': 'August 28, 2026', 'found': 'Yes', 'keyword': 'Piano Tuning'}

In [13]:
res["bid_due_date"]

'10/29/2026'

In [18]:
with pymupdf.open(pdf_path) as doc:
    for n, page in enumerate(doc, 1):
        text = norm(page.get_text())
        print(f"page {n}: chars={len(text)}, keyword={bool(KEYWORD_RE.search(text))}, dates={len(date_candidates(text))}")

page 1: chars=1753, keyword=True, dates=6
page 2: chars=199, keyword=False, dates=1


In [19]:
with pymupdf.open(pdf_path) as doc:
    text = norm(doc[0].get_text())

for date, context in date_candidates(text):
    print(is_due_date(context), "|", context)

False | #1 Changes and/or Additions issued on [[October 2, 2026]]. Description:
True | virtually using Microsoft Teams on [[September 24, 2026]], at 2 PM. to
True | #1 Changes and/or Additions issued on [[October 2, 2026]]. Publication
True | 2, 2026. Publication Date/Time: [[9/15/2026]] 10:00 AM
True | for Proposal Closing Date/Time: [[10/13/2026]] 4:00 PM
True | Date/Time: Addendum #1 issued on [[October 2, 2026]]. Contact
